# 03 - RMSNorm

In the previous notebook, we built a baseline GPT architecture using **Layer Normalization**.

TinyLlama replaces LayerNorm with **Root Mean Square Layer Normalization (RMSNorm)** because it is computationally cheaper while maintaining similar performance.

## Objectives

In this notebook we will:

- Understand RMSNorm
- Implement RMSNorm from scratch
- Compare it with LayerNorm
- Replace LayerNorm in the GPT architecture
- Verify the updated architecture
- Save the RMSNorm model for future notebooks

#Imports & Drive Mount

In [ ]:
import os
import json
from pathlib import Path

import torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass

# Mount Google Drive (only if not already mounted)
if not os.path.exists("/content/drive/MyDrive"):
    from google.colab import drive
    drive.mount("/content/drive")

# Project directories
ROOT = Path("/content/drive/MyDrive/TinyLlama-repo")

CONFIG_DIR = ROOT / "config"
CHECKPOINTS_DIR = ROOT / "checkpoints"
OUTPUTS_DIR = ROOT / "outputs"
SRC_DIR = ROOT / "src"

# Create directories if they don't exist
for dir in [CONFIG_DIR, CHECKPOINTS_DIR, OUTPUTS_DIR, SRC_DIR]:
    if not os.path.exists(dir):
        os.makedirs(dir)

# Device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Load GPT configuration
with open(CONFIG_DIR / "gpt_config.json", "r") as f:
    config_dict = json.load(f)

print("Configuration loaded successfully.")
print(config_dict)

@dataclass
class GPTConfig:
    vocab_size: int
    max_seq_len: int
    d_model: int
    n_heads: int
    n_layers: int
    dropout: float
    bias: bool

config = GPTConfig(**config_dict)

print("Configuration loaded successfully.")
print(config)

Using device: cuda
Configuration loaded successfully.
{'vocab_size': 4096, 'max_seq_len': 512, 'd_model': 512, 'n_heads': 8, 'n_layers': 6, 'dropout': 0.1, 'bias': False}
Configuration loaded successfully.
GPTConfig(vocab_size=4096, max_seq_len=512, d_model=512, n_heads=8, n_layers=6, dropout=0.1, bias=False)


#Load Baseline Model

In [ ]:
import sys

# Add project root to Python path
if str(ROOT) not in sys.path:
    sys.path.append(str(ROOT))

# Import baseline GPT architecture
from src.gpt_baseline import GPT

# Create the baseline model
model = GPT(config).to(device)

# Load baseline weights
checkpoint_path = CHECKPOINTS_DIR / "gpt_baseline.pt"

checkpoint = torch.load(
    checkpoint_path,
    map_location=device
)

# Load state dictionary
model.load_state_dict(checkpoint)

# Set model to evaluation mode
model.eval()

print("Baseline GPT loaded successfully!")
print(f"Checkpoint loaded from: {checkpoint_path}")

Baseline GPT loaded successfully!
Checkpoint loaded from: /content/drive/MyDrive/TinyLlama-repo/checkpoints/gpt_baseline.pt


# What is RMSNorm?

LayerNorm performs two operations:

1. Centers the data (subtracts the mean)
2. Normalizes the variance

RMSNorm simplifies this.

Instead of subtracting the mean, RMSNorm only scales the vector using its Root Mean Square (RMS).

This reduces computation while preserving training stability.

TinyLlama, LLaMA, Mistral and many modern LLMs use RMSNorm instead of LayerNorm.

#RMSNorm

In [ ]:
class RMSNorm(nn.Module):

    def __init__(
        self,
        dim,
        eps=1e-6
    ):

        super().__init__()

        self.weight = nn.Parameter(torch.ones(dim))

        self.eps = eps

    def forward(self, x):

        rms = torch.rsqrt(
            x.pow(2).mean(-1, keepdim=True) + self.eps
        )

        return self.weight * x * rms

#Test RMSNorm

In [ ]:
rms = RMSNorm(config.d_model)

x = torch.randn(2, 16, config.d_model)

out = rms(x)

print("Input :", x.shape)

print("Output:", out.shape)

Input : torch.Size([2, 16, 512])
Output: torch.Size([2, 16, 512])


Expected
```
Input : torch.Size([2,10,512])

Output: torch.Size([2,10,512])

# Compare RMSNorm and LayerNorm

Both layers preserve the tensor shape.

The primary difference is the normalization strategy.

This comparison is only to verify that both layers operate on tensors of the same dimensions.

In [ ]:
layer_norm = nn.LayerNorm(config.d_model)

rms_norm = RMSNorm(config.d_model)

x = torch.randn(
    2,
    16,
    config.d_model
)

layer_out = layer_norm(x)

rms_out = rms_norm(x)

print("LayerNorm :", layer_out.shape)

print("RMSNorm   :", rms_out.shape)

LayerNorm : torch.Size([2, 16, 512])
RMSNorm   : torch.Size([2, 16, 512])


Expected
```
torch.Size([2,16,512])

torch.Size([2,16,512])

#Parameter Comparison

In [ ]:
ln_params = sum(

    p.numel()

    for p in layer_norm.parameters()

)

rms_params = sum(

    p.numel()

    for p in rms_norm.parameters()

)

print("LayerNorm Parameters :", ln_params)

print("RMSNorm Parameters   :", rms_params)

LayerNorm Parameters : 1024
RMSNorm Parameters   : 512


Expected
```
LayerNorm : 1024

RMSNorm : 512
```
This is one reason RMSNorm is preferred, it has fewer learnable parameters because it omits the bias term.

# Replacing LayerNorm

Our GPT model contains LayerNorm in every decoder block.

We now replace each LayerNorm with RMSNorm while keeping the rest of the architecture unchanged.

In [ ]:
def replace_layernorm(module):

    for name, child in module.named_children():

        if isinstance(child, nn.LayerNorm):

            # Create an RMSNorm layer with the same embedding dimension
            rmsnorm = RMSNorm(
                dim=child.normalized_shape[0],
                eps=child.eps
            )

            # Copy the learned scaling parameter (gamma)
            with torch.no_grad():
                rmsnorm.weight.copy_(child.weight)

            # Replace LayerNorm with RMSNorm
            setattr(module, name, rmsnorm)

        else:
            replace_layernorm(child)

#Apply Replacement

In [ ]:
replace_layernorm(model)

model = model.to(device)  # new RMSNorm layers default to CPU

print(model)

GPT(
  (token_embedding): TokenEmbedding(
    (embedding): Embedding(4096, 512)
  )
  (position_embedding): PositionalEmbedding(
    (embedding): Embedding(512, 512)
  )
  (blocks): ModuleList(
    (0-5): 6 x DecoderBlock(
      (ln1): RMSNorm()
      (attn): MultiHeadAttention(
        (q): Linear(in_features=512, out_features=512, bias=False)
        (k): Linear(in_features=512, out_features=512, bias=False)
        (v): Linear(in_features=512, out_features=512, bias=False)
        (out): Linear(in_features=512, out_features=512, bias=False)
      )
      (ln2): RMSNorm()
      (ffn): FeedForward(
        (net): Sequential(
          (0): Linear(in_features=512, out_features=2048, bias=False)
          (1): GELU(approximate='none')
          (2): Linear(in_features=2048, out_features=512, bias=False)
          (3): Dropout(p=0.1, inplace=False)
        )
      )
    )
  )
  (final_norm): RMSNorm()
  (lm_head): Linear(in_features=512, out_features=4096, bias=False)
)


In [ ]:
layernorm_count = 0
rmsnorm_count = 0

for module in model.modules():
    if isinstance(module, nn.LayerNorm):
        layernorm_count += 1
    elif isinstance(module, RMSNorm):
        rmsnorm_count += 1

print(f"LayerNorm Layers Remaining : {layernorm_count}")
print(f"RMSNorm Layers             : {rmsnorm_count}")

assert layernorm_count == 0, "Some LayerNorm layers were not replaced."

print("All LayerNorm layers successfully replaced.")

LayerNorm Layers Remaining : 0
RMSNorm Layers             : 13
All LayerNorm layers successfully replaced.


#Forward Pass

In [ ]:
x = torch.randint(

    0,

    config.vocab_size,

    (2,128)

).to(device)

with torch.no_grad():

    logits = model(x)

print(logits.shape)

torch.Size([2, 128, 4096])


Expeceted
```
torch.Size([2,128,32000])

#Export RMSNorm Model Class

In [ ]:
gpt_rmsnorm_code = r'''
import math
import torch
import torch.nn as nn
import torch.nn.functional as F


class TokenEmbedding(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.embedding = nn.Embedding(
            config.vocab_size,
            config.d_model
        )

    def forward(self, x):
        return self.embedding(x)


class PositionalEmbedding(nn.Module):

    def __init__(self, config):
        super().__init__()
        self.embedding = nn.Embedding(
            config.max_seq_len,
            config.d_model
        )

    def forward(self, x):

        _, seq_len = x.shape

        positions = torch.arange(
            seq_len,
            device=x.device
        ).unsqueeze(0)

        return self.embedding(positions)


def create_causal_mask(seq_len):

    return torch.tril(torch.ones(seq_len, seq_len))


def split_heads(x, n_heads):

    batch_size, seq_len, d_model = x.shape

    head_dim = d_model // n_heads

    x = x.view(
        batch_size,
        seq_len,
        n_heads,
        head_dim
    )

    return x.transpose(1,2)


def merge_heads(x):

    batch_size, heads, seq_len, head_dim = x.shape

    x = x.transpose(1,2).contiguous()

    return x.view(
        batch_size,
        seq_len,
        heads * head_dim
    )


class RMSNorm(nn.Module):

    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        rms = torch.rsqrt(
            x.pow(2).mean(-1, keepdim=True) + self.eps
        )
        return self.weight * x * rms


class MultiHeadAttention(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.n_heads = config.n_heads
        self.head_dim = config.d_model // config.n_heads

        self.q = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.k = nn.Linear(config.d_model, config.d_model, bias=config.bias)
        self.v = nn.Linear(config.d_model, config.d_model, bias=config.bias)

        self.out = nn.Linear(config.d_model, config.d_model, bias=config.bias)

    def forward(self,x):

        q = split_heads(self.q(x), self.n_heads)
        k = split_heads(self.k(x), self.n_heads)
        v = split_heads(self.v(x), self.n_heads)

        scores = torch.matmul(
            q,
            k.transpose(-2,-1)
        )

        scores = scores / math.sqrt(self.head_dim)

        mask = create_causal_mask(
            scores.size(-1)
        ).to(scores.device)

        scores = scores.masked_fill(
            mask==0,
            float("-inf")
        )

        weights = F.softmax(
            scores,
            dim=-1
        )

        out = torch.matmul(weights,v)

        out = merge_heads(out)

        return self.out(out)


class FeedForward(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.net = nn.Sequential(
            nn.Linear(
                config.d_model,
                4*config.d_model,
                bias=config.bias
            ),
            nn.GELU(),
            nn.Linear(
                4*config.d_model,
                config.d_model,
                bias=config.bias
            ),
            nn.Dropout(config.dropout)
        )

    def forward(self,x):

        return self.net(x)


class DecoderBlock(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.ln1 = RMSNorm(config.d_model)

        self.attn = MultiHeadAttention(config)

        self.ln2 = RMSNorm(config.d_model)

        self.ffn = FeedForward(config)

    def forward(self,x):

        x = x + self.attn(self.ln1(x))

        x = x + self.ffn(self.ln2(x))

        return x


class GPT(nn.Module):

    def __init__(self, config):

        super().__init__()

        self.token_embedding = TokenEmbedding(config)

        self.position_embedding = PositionalEmbedding(config)

        self.blocks = nn.ModuleList(
            [
                DecoderBlock(config)
                for _ in range(config.n_layers)
            ]
        )

        self.final_norm = RMSNorm(config.d_model)

        self.lm_head = nn.Linear(
            config.d_model,
            config.vocab_size,
            bias=False
        )

    def forward(self,input_ids):

        x = self.token_embedding(input_ids)

        x = x + self.position_embedding(input_ids)

        for block in self.blocks:

            x = block(x)

        x = self.final_norm(x)

        return self.lm_head(x)
'''

with open(SRC_DIR / "gpt_rmsnorm.py", "w") as f:
    f.write(gpt_rmsnorm_code)

print("gpt_rmsnorm.py saved successfully.")

gpt_rmsnorm.py saved successfully.


In [ ]:
# Re-import from the file we just wrote, transfer weights from the
# in-memory (runtime-replaced) model, and validate keys match exactly.
import importlib

import src.gpt_rmsnorm as gpt_rmsnorm_module
importlib.reload(gpt_rmsnorm_module)

model_final = gpt_rmsnorm_module.GPT(config).to(device)
model_final.load_state_dict(model.state_dict())  # raises if keys/shapes mismatch
model_final.eval()

print("model_final built from src/gpt_rmsnorm.py and weights transferred.")

model_final built from src/gpt_rmsnorm.py and weights transferred.


#Save Updated Weights

In [ ]:
CHECKPOINTS_DIR.mkdir(parents=True, exist_ok=True)

rmsnorm_ckpt = CHECKPOINTS_DIR / "gpt_rmsnorm.pt"
current_keys = set(model_final.state_dict().keys())

def _checkpoint_keys_match(path):
    try:
        ck = torch.load(path, map_location="cpu")
        return set(ck.keys()) == current_keys
    except Exception:
        return False

if rmsnorm_ckpt.exists() and _checkpoint_keys_match(rmsnorm_ckpt):
    print("Checkpoint exists and matches current model, skip:", rmsnorm_ckpt)
else:
    if rmsnorm_ckpt.exists():
        print("Checkpoint exists but keys mismatch current model -- overwriting.")
    fp16_state = {k: v.half() for k, v in model_final.state_dict().items()}
    torch.save(fp16_state, rmsnorm_ckpt)
    print("Saved:", rmsnorm_ckpt)

Checkpoint exists and matches current model, skip: /content/drive/MyDrive/TinyLlama-repo/checkpoints/gpt_rmsnorm.pt


#Save Summary

In [ ]:
summary = {
    "base_model": "gpt_baseline.pt",
    "updated_model": "gpt_rmsnorm.pt",
    "model_class_file": "src/gpt_rmsnorm.py",
    "normalization": "RMSNorm",
    "layernorm_parameters": ln_params,
    "rmsnorm_parameters": rms_params,
    "total_parameters": sum(
        p.numel() for p in model.parameters()
    ),
    "forward_pass": "passed"
}

with open(
    OUTPUTS_DIR / "rmsnorm_summary.json",
    "w"
) as f:
    json.dump(summary, f, indent=4)

#Final Verification

In [ ]:
assert logits.shape == (

    2,

    128,

    config.vocab_size

)

print("""
=====================================

RMSNorm Successfully Integrated

Artifacts Saved

✓ gpt_rmsnorm.pt

✓ rmsnorm_summary.json

Ready for Notebook 4 (RoPE)

=====================================
""")



RMSNorm Successfully Integrated

Artifacts Saved

✓ gpt_rmsnorm.pt

✓ rmsnorm_summary.json

Ready for Notebook 4 (RoPE)




In [ ]:
architecture_test = {
    "input_shape": list(x.shape),
    "output_shape": list(logits.shape),
    "layernorm_remaining": layernorm_count,
    "rmsnorm_layers": rmsnorm_count,
    "forward_pass": "passed"
}

with open(
    OUTPUTS_DIR / "rmsnorm_architecture_test.json",
    "w"
) as f:
    json.dump(architecture_test, f, indent=4)